# Lab 5 — N-gram Language Models

In [3]:
import re, math, random
from collections import Counter

import nltk
nltk.download('gutenberg', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import gutenberg


## Корпус і токенізація

In [4]:
raw_sents = list(gutenberg.sents('melville-moby_dick.txt'))

def clean(sent):
    return [w.lower() for w in sent if re.match(r"[a-z']+", w.lower())]

sentences = [clean(s) for s in raw_sents if clean(s)]

random.seed(42)
random.shuffle(sentences)

split = int(0.8 * len(sentences))
train = sentences[:split]
test  = sentences[split:]

print(f"Всього речень: {len(sentences)}")
print(f"Train: {len(train)}, Test: {len(test)}")
print(f"Приклад: {train[0][:10]}")


Всього речень: 10031
Train: 8024, Test: 2007
Приклад: ['look', 'ye']


## Task 0 — N-gram модель


In [5]:
class NGramModel:
    def __init__(self, n, laplace=False):
        self.n = n
        self.laplace = laplace
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()

    def train(self, sentences):
        for sent in sentences:
            padded = ['<s>'] * (self.n - 1) + sent + ['</s>']
            self.vocab.update(sent)
            for i in range(len(padded) - self.n + 1):
                ngram = tuple(padded[i:i + self.n])
                self.ngram_counts[ngram] += 1
                self.context_counts[ngram[:-1]] += 1
        self.vocab.update(['<s>', '</s>'])

    def prob(self, word, context):
        ngram = context + (word,)
        c_ngram   = self.ngram_counts[ngram]
        c_context = self.context_counts[context]
        V = len(self.vocab)
        if self.laplace:
            return (c_ngram + 1) / (c_context + V)
        return c_ngram / c_context if c_context > 0 else 0.0

    def sentence_log_prob(self, sentence):
        padded = ['<s>'] * (self.n - 1) + sentence + ['</s>']
        log_p = 0.0
        for i in range(self.n - 1, len(padded)):
            context = tuple(padded[i - self.n + 1:i])
            p = self.prob(padded[i], context)
            if p <= 0:
                return float('-inf')
            log_p += math.log(p)
        return log_p

    def perplexity(self, sentences):
        total_log_p, total_words = 0.0, 0
        for sent in sentences:
            lp = self.sentence_log_prob(sent)
            if lp == float('-inf'):
                return float('inf')
            total_log_p += lp
            total_words += len(sent) + 1
        return math.exp(-total_log_p / total_words)

    def next_word_probs(self, context):
        return {w: self.prob(w, context) for w in self.vocab if self.prob(w, context) > 0}


In [6]:
bigram  = NGramModel(n=2, laplace=True)
trigram = NGramModel(n=3, laplace=True)
bigram.train(train)
trigram.train(train)

print(f"Словник: {len(bigram.vocab)} слів")
print(f"Біграм унікальних: {len(bigram.ngram_counts)}")
print(f"Триграм унікальних: {len(trigram.ngram_counts)}")

print("\nP(w | 'the') — топ-6:")
for w, p in sorted(bigram.next_word_probs(('the',)).items(), key=lambda x: -x[1])[:6]:
    print(f"  {w}: {p:.4f}")


Словник: 15298 слів
Біграм унікальних: 95838
Триграм унікальних: 154380

P(w | 'the') — топ-6:
  whale: 0.0123
  ship: 0.0074
  sea: 0.0069
  same: 0.0050
  pequod: 0.0048
  white: 0.0044


## Task 1 — Bigram vs Trigram


In [7]:
ppl_bi  = bigram.perplexity(test)
ppl_tri = trigram.perplexity(test)

print(f"Bigram  perplexity: {ppl_bi:.2f}")
print(f"Trigram perplexity: {ppl_tri:.2f}")
print(f"Краща: {'bigram' if ppl_bi < ppl_tri else 'trigram'}")


Bigram  perplexity: 3544.03
Trigram perplexity: 10065.00
Краща: bigram


In [8]:
examples = [
    ["the", "whale", "was", "large"],
    ["call", "me", "ishmael"],
    ["the", "sea", "was", "calm"],
]

print(f"{'речення':<35} {'bigram':>10} {'trigram':>10}")
print("-" * 57)
for sent in examples:
    lp_bi  = bigram.sentence_log_prob(sent)
    lp_tri = trigram.sentence_log_prob(sent)
    print(f"  {' '.join(sent):<33} {lp_bi:>10.3f} {lp_tri:>10.3f}")


речення                                 bigram    trigram
---------------------------------------------------------
  the whale was large                  -33.448    -37.225
  call me ishmael                      -35.313    -37.578
  the sea was calm                     -35.324    -39.923


## Task 2 — Stupid Backoff і Interpolation


In [9]:
class StupidBackoff:
    def __init__(self, n=3, lam=0.4):
        self.n = n
        self.lam = lam
        self.models = [NGramModel(k) for k in range(1, n + 1)]
        self.vocab = set()

    def train(self, sentences):
        for m in self.models:
            m.train(sentences)
        self.vocab = self.models[-1].vocab

    def score(self, word, context, order=None):
        if order is None:
            order = self.n
        if order == 1:
            m = self.models[0]
            total = sum(m.ngram_counts.values())
            V = len(self.vocab)
            return (m.ngram_counts[(word,)] + 1) / (total + V)
        m = self.models[order - 1]
        ctx = context[-(order - 1):]
        if m.ngram_counts[ctx + (word,)] > 0:
            return m.ngram_counts[ctx + (word,)] / m.context_counts[ctx]
        return self.lam * self.score(word, context, order - 1)

    def sentence_log_prob(self, sentence):
        padded = ['<s>'] * (self.n - 1) + sentence + ['</s>']
        log_p = 0.0
        for i in range(self.n - 1, len(padded)):
            ctx = tuple(padded[i - self.n + 1:i])
            log_p += math.log(max(self.score(padded[i], ctx), 1e-300))
        return log_p

    def perplexity(self, sentences):
        total_log_p, total_words = 0.0, 0
        for sent in sentences:
            total_log_p += self.sentence_log_prob(sent)
            total_words += len(sent) + 1
        return math.exp(-total_log_p / total_words)

    def next_word_probs(self, context):
        return {w: self.score(w, context) for w in self.vocab}


In [10]:
class Interpolation:
    def __init__(self, n=3, lambdas=None):
        self.n = n
        self.lambdas = lambdas or [1/n] * n
        self.models = [NGramModel(k) for k in range(1, n + 1)]
        self.vocab = set()

    def train(self, sentences):
        for m in self.models:
            m.train(sentences)
        self.vocab = self.models[-1].vocab

    def prob(self, word, context):
        p = 0.0
        V = len(self.vocab)
        for i, m in enumerate(self.models):
            ctx = context[-i:] if i > 0 else ()
            if i == 0:
                total = sum(m.ngram_counts.values())
                pi = (m.ngram_counts[(word,)] + 1) / (total + V)
            else:
                pi = m.prob(word, ctx)
            p += self.lambdas[i] * pi
        return p

    def sentence_log_prob(self, sentence):
        padded = ['<s>'] * (self.n - 1) + sentence + ['</s>']
        log_p = 0.0
        for i in range(self.n - 1, len(padded)):
            ctx = tuple(padded[i - self.n + 1:i])
            log_p += math.log(max(self.prob(padded[i], ctx), 1e-300))
        return log_p

    def perplexity(self, sentences):
        total_log_p, total_words = 0.0, 0
        for sent in sentences:
            total_log_p += self.sentence_log_prob(sent)
            total_words += len(sent) + 1
        return math.exp(-total_log_p / total_words)

    def next_word_probs(self, context):
        return {w: self.prob(w, context) for w in self.vocab}


In [11]:
backoff = StupidBackoff(n=3)
interp  = Interpolation(n=3, lambdas=[0.1, 0.3, 0.6])
backoff.train(train)
interp.train(train)

print(f"{'Модель':<20} {'PPL (test)':>10}")
print("-" * 32)
for name, m in [("Bigram+Laplace", bigram), ("Trigram+Laplace", trigram),
                ("Stupid Backoff", backoff), ("Interpolation",  interp)]:
    print(f"  {name:<18} {m.perplexity(test):>10.2f}")


Модель               PPL (test)
--------------------------------
  Bigram+Laplace        3544.03
  Trigram+Laplace      10065.00
  Stupid Backoff         749.82
  Interpolation         1018.73


In [12]:
oov = ["the", "ocean", "was", "dark"]
print(f"Речення: '{' '.join(oov)}'")
for name, m in [("Bigram+Laplace", bigram), ("Stupid Backoff", backoff), ("Interpolation", interp)]:
    print(f"  {name}: log P = {m.sentence_log_prob(oov):.3f}")


Речення: 'the ocean was dark'
  Bigram+Laplace: log P = -38.538
  Stupid Backoff: log P = -29.982
  Interpolation: log P = -30.669


## Task 3 — Генерація тексту


In [13]:
def generate(model, prompt, max_words=15, temperature=0.8, seed=None):
    if seed is not None:
        random.seed(seed)
    n = model.n
    result = list(prompt)
    for _ in range(max_words):
        context = tuple((['<s>'] * (n - 1) + result)[-(n - 1):])
        dist = model.next_word_probs(context)
        dist.pop('</s>', None)
        dist.pop('<s>',  None)
        if not dist:
            break
        words  = list(dist)
        scores = [dist[w] ** (1 / temperature) for w in words]
        total  = sum(scores)
        probs  = [s / total for s in scores]
        result.append(random.choices(words, weights=probs)[0])
    return ' '.join(result)


In [14]:
prompts = [["the", "whale"], ["call", "me"], ["the", "sea"], ["a", "ship"]]

for name, m in [("Bigram", bigram), ("Interpolation", interp), ("Backoff", backoff)]:
    print(f"[{name}]")
    for p in prompts:
        print(f"  {p} -> {generate(m, p, seed=7)}")
    print()


[Bigram]
  ['the', 'whale'] -> the whale dismemberer anvil thundering phantoms sixteenth official plaid wages kidnapped rich unpublished ascribe topmost statue languid
  ['call', 'me'] -> call me magnetic fitness thundering phantoms sixteenth official plaid wages kidnapped rich unpublished ascribe topmost statue languid
  ['the', 'sea'] -> the sea mab anvil thundering phantoms sixteenth official plaid wages kidnapped rich unpublished ascribe topmost statue languid
  ['a', 'ship'] -> a ship intolerableness anvil thundering phantoms sixteenth official plaid wages kidnapped rich unpublished ascribe topmost statue languid

[Interpolation]
  ['the', 'whale'] -> the whale ship ' s fish an old gay head to make any very easy matter for
  ['call', 'me'] -> call me an empty vial even then beyond then the wild business that for some one '
  ['the', 'sea'] -> the sea for by the french and the hum of human grandeur beyond which few mortals will
  ['a', 'ship'] -> a ship ' s the mighty difference be

In [15]:
print("Вплив temperature (Interpolation, 'the whale'):")
for T in [0.3, 0.8, 1.5]:
    print(f"  T={T}: {generate(interp, ['the', 'whale'], temperature=T, seed=42)}")


Вплив temperature (Interpolation, 'the whale'):
  T=0.3: the whale ' s the old man ' s the matter of the whale ' s a
  T=0.8: the whale and often in what it is but there is called proves an instant in a
  T=1.5: the whale towing often decided there weekly had seat english resident sworn odor compacted collectedness column the
